<a href="https://colab.research.google.com/github/soule-geophysics/geol-333-714/blob/main/notebooks/week_02_lsq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> **Do this first: File > Save a copy in Drive.**
> You are viewing a shared notebook. Anything you type here is NOT saved.
> Save your own copy now and work only in the copy. We work this notebook
> together in class; nothing here is submitted.

> **Colab deletes uploaded files when the runtime disconnects.**
> If a CSV you uploaded by hand has vanished, re-run the data-loading cell above
> (the URL load restores the data) or re-upload the file. Code and written
> answers persist in your own saved copy.

In [ ]:
import pandas as pd
import plotly.express as px
import numpy as np

# Course accessibility default: colorblind-safe qualitative palette
px.defaults.color_discrete_sequence = px.colors.qualitative.Safe

# Primary data path: stable public URL (no upload needed).
PENDULUM_URL = "https://raw.githubusercontent.com/soule-geophysics/geol-333-714/main/data/pendulum_sample.csv"
STAIRWELL_URL = "https://raw.githubusercontent.com/soule-geophysics/geol-333-714/main/data/stairwell.csv"

# Fallback: download both CSVs from the Week 2 Guide on Brightspace, drag them
# into the Colab Files panel, and use the bare filenames instead (uncomment the
# two lines below, comment the two above):
# PENDULUM_URL = "pendulum_sample.csv"
# STAIRWELL_URL = "stairwell.csv"

pendulum = pd.read_csv(PENDULUM_URL)
stairwell = pd.read_csv(STAIRWELL_URL)

print('pendulum:', pendulum.shape, '| stairwell:', stairwell.shape)

## Data sources and citation

**Stairwell gravimeter data** (used in Parts 6–9) comes from a published teaching collection:

> Parsekian, A. (n.d.). *IGUaNA Unit 3: Gravity and Magnetics Field Data Exercises, Part 3a (Stairwell gravity).* Science Education Resource Center, Carleton College. CC-BY-NC-SA 4.0. <https://serc.carleton.edu/iguana/teaching_materials/grav_mag/unit3.html>

These are real gravimeter readings taken in a campus stairwell: open-and-close ground-level reads to track instrument drift, plus one reading at each half-floor as the operator climbed.

**Pendulum data** (used in Parts 2–5) is a synthetic dataset generated with a fixed random seed to mimic the course demo apparatus: each trial times 10 complete swings and divides by 10, amplitude under 15 degrees, 5 lengths from 0.4 to 1.2 m (true pendulum lengths, pivot to the center of the bob), 5 trials per length. No external license applies.

### Getting unstuck

Work this ladder in order:

1. Re-read this section's markdown, then **Runtime > Run all above** (cells must run in order).
2. Check the previous self-check cell. If your value is out of range, the problem is upstream of where it surfaced.
3. Ask in class, or post on the **Ask the Class (General Q&A)** discussion topic.
4. Paste your own code and its error message into CUNY Copilot (microsoft365.com/chat, CUNY Login) to understand why it fails. Asking Copilot to write the solution for you is prohibited; asking it why your line fails is permitted.

# Week 2: PDFs and the Method of Least Squares

**Course:** GEOL 333 / 714, Geophysical Exploration Methods, Fall 2026
**Instructor:** Dax Soule (dax.soule@qc.cuny.edu)
**Meeting:** Wednesday, September 9, 2026, in person

We work this notebook together on the projector; you follow along in your own saved copy.

## What tonight does

Tonight formalizes the straight-line fit you did by eye in Week 1. This notebook re-fits the pendulum data in matrix form, previews the stairwell drift and free-air fits behind HW1, and closes with three buggy drift-correction snippets to critique after class. Least squares extracts three different numbers from the world of `g = GM/R²`: `g` itself (the pendulum slope), the instrument drift the equation assumes away (the stairwell base ties), and how fast `g` decreases with height above `R` (the free-air gradient).

## Part 1: Matrix warm-up, systems and determinants

Least squares reduces to a system of two equations in two unknowns. Here is one with a clean answer:

$$2x + y = 5$$
$$x + 3y = 10$$

Write it as $A\,\mathbf{x} = \mathbf{b}$ with

$$A = \begin{bmatrix} 2 & 1 \\ 1 & 3 \end{bmatrix}, \qquad \mathbf{b} = \begin{bmatrix} 5 \\ 10 \end{bmatrix}.$$

The **determinant** $\det A = ad - bc = (2)(3) - (1)(1) = 5$. A nonzero determinant means the two lines cross at exactly one point, so the system has a unique solution. The cell below solves it.

This week's single method, least squares, extracts three different numbers from the world of `g = GM/R²`: g itself (the pendulum slope), the instrument drift the equation assumes away (the stairwell base ties), and how fast g decreases with height above R (the free-air gradient).

In [ ]:
# Solve the 2x2 system 2x + y = 5, x + 3y = 10.
A = np.array([[2.0, 1.0],
              [1.0, 3.0]])
b = np.array([5.0, 10.0])

det_A = np.linalg.det(A)
solution = np.linalg.solve(A, b)

print(f'determinant: {det_A:.4f}')
print(f'solution: x = {solution[0]:.4f}, y = {solution[1]:.4f}')

Now a counter-example. Change the second row so it is just twice the first:

$$A_{\text{singular}} = \begin{bmatrix} 1 & 2 \\ 2 & 4 \end{bmatrix}, \qquad \det = (1)(4) - (2)(2) = 0.$$

When $\det = 0$ the two equations describe the **same line**, so there is no single crossing point: the system is *singular* and has no unique answer. This is the seed of an inverse problem, where many different models can fit the same data equally well. Run the cell; the solver raises LinAlgError.

In [ ]:
# A singular system: the second equation is twice the first.
A_singular = np.array([[1.0, 2.0],
                       [2.0, 4.0]])
b_singular = np.array([3.0, 6.0])

print(f'determinant: {np.linalg.det(A_singular):.4f}')

try:
    np.linalg.solve(A_singular, b_singular)
except np.linalg.LinAlgError as err:
    print(f'solver raised LinAlgError: {err}')
    print('A zero determinant means no unique solution (non-uniqueness).')

**Self-check.** Your determinant should equal **5** and your solution should be **x = 1, y = 3**. If your determinant is 7, you added the off-diagonal product instead of subtracting it ($ad - bc$, not $ad + bc$).

## Part 2: HW0 debrief, the pendulum answer

HW0 is due next Wednesday, Sep 16, 11:59 PM, to the HW0 dropbox. Tonight's debrief walks the method on the sample CSV.

On the sample CSV, fitting $T^2$ against $l$ gives a slope near 4.086 s²/m, and $g = 4\pi^2 / \text{slope} \approx 9.66$ m/s². That is about 1.5% below the textbook 9.81 m/s². That is an ordinary noise draw: the periods carry stopwatch scatter, and a single 25-trial sample lands near the true value. HW0's method: time 10 swings, divide by 10, stack trials, fit, report uncertainty. Re-run the fit below.

In [ ]:
# Re-run the HW0 pendulum fit: T^2 vs l, slope -> g.
pendulum['T2'] = pendulum['period_s'] ** 2
slope, intercept = np.polyfit(pendulum['length_m'], pendulum['T2'], 1)

# 1-sigma on the slope, propagated to g = 4 pi^2 / slope.
_, cov = np.polyfit(pendulum['length_m'], pendulum['T2'], 1, cov=True)
slope_sigma = np.sqrt(cov[0, 0])
g = 4 * np.pi**2 / slope
g_sigma = 4 * np.pi**2 / slope**2 * slope_sigma

print(f'slope:     {slope:.4f} s^2/m')
print(f'g:         {g:.2f} +/- {g_sigma:.2f} m/s^2')
print(f'textbook:  9.81 m/s^2')

**Self-check.** Expected: slope **4.086 s²/m**, g = **9.66 ± 0.08 m/s²**. If your slope is near 1.2, you fitted $T$ vs $l$; square the period first.

## Part 3: Histograms to PDFs

Each pendulum length has a *stack* of timed trials (5 per length in the sample CSV), and each trial is already an average of 10 swings. A stack of repeated measurements has a shape: most values cluster near the middle, fewer land in the tails. As the number of trials grows, the histogram of a stack approaches a smooth curve, the [**probability density function (PDF)**](https://github.com/soule-geophysics/geol-333-714/blob/main/GLOSSARY.md#probability-density-function-pdf). The cell below summarizes every length's stack with `groupby`, then histograms the l = 1.0 m stack.

In [ ]:
# Per-length summary statistics on the period stacks.
stack_stats = pendulum.groupby('length_m')['period_s'].agg(['mean', 'std', 'count'])
print(stack_stats.round(4))

In [ ]:
# Histogram of the l = 1.0 m stack.
stack_10 = pendulum[pendulum['length_m'] == 1.0]
fig = px.histogram(
    stack_10,
    x='period_s',
    nbins=5,
    title='Period stack at l = 1.0 m',
    labels={'period_s': 'Period T (s)', 'count': 'Number of trials'},
)
fig.show()

print(f"L = 1.0 m stack: mean {stack_10['period_s'].mean():.3f} s, "
      f"std {stack_10['period_s'].std():.3f} s, count {len(stack_10)}")

**Figure description:** A histogram of the 5 period measurements at L = 1.0 m, with period (seconds) on the x-axis and trial count on the y-axis. The five trials cluster near 2.014 s with a spread of about 0.011 s; with only 5 trials the bars are coarse, but the central clustering is the seed of the bell-shaped PDF the next part overlays. The mean, standard deviation, and count are printed by the cell so the numbers are available without rendering the plot.

**Self-check.** At L = 1.0 m: mean **2.014 s**, std **0.011 s**, count **5**. If your std is ten times larger (around 0.1 s), you pooled all lengths into one stack instead of grouping by length.

## Part 4: The normal curve and the 68-95-99.7 rule

A single 5-trial stack is too small to look like a bell on its own. But if we **standardize** every trial against *its own length's* mean and standard deviation, $z = (T - \mu_L) / \sigma_L$, then all trials live on one common axis (25 in the sample CSV) and we can ask whether they follow the normal curve. For the normal PDF, about **68%** of values fall within one standard deviation of the mean, **95%** within two, and **99.7%** within three. The cell below builds the 25 z-scores and counts how many land inside one standard deviation.

In [ ]:
# Standardize each trial against ITS OWN length's mean and std.
length_mean = pendulum.groupby('length_m')['period_s'].transform('mean')
length_std = pendulum.groupby('length_m')['period_s'].transform('std')
pendulum['z'] = (pendulum['period_s'] - length_mean) / length_std

within_one_sigma = (pendulum['z'].abs() <= 1).sum()
n_total = len(pendulum)

fig = px.histogram(
    pendulum,
    x='z',
    nbins=10,
    title='Standardized periods (25 z-scores), all lengths pooled',
    labels={'z': 'z = (T - length mean) / length std', 'count': 'Number of trials'},
)
fig.show()

print(f'{within_one_sigma} of {n_total} trials '
      f'({100 * within_one_sigma / n_total:.0f}%) fall within one std of their own mean.')
print('The 68-95-99.7 rule predicts about 68% within one std.')

**Figure description:** A histogram of the 25 standardized periods (z-scores), centered on zero, with z on the x-axis and trial count on the y-axis. The values pile up near zero and thin out past |z| = 1, the shape the normal PDF predicts; 16 of 25 (64%) land within one standard deviation. The count is printed by the cell so the takeaway does not depend on reading the plot.

The standard error $\text{SE} = \sigma / \sqrt{N}$ is a statement about the PDF of the *mean*. Stacking 5 trials narrows that PDF by a factor of $\sqrt{5}$, which is why you stacked trials instead of trusting one timing.

**Self-check.** **16 of the 25 trials (64%)** fall within one std of their own length's mean; the normal rule predicts 68%, and with N = 25 anything from about 14 to 20 is unremarkable. If you got far fewer, check that you used each length's own mean, not the global mean across all lengths.

## Part 5: Least squares, derived and re-fit by matrix

A best-fit line $y = m x + c$ leaves a [**residual**](https://github.com/soule-geophysics/geol-333-714/blob/main/GLOSSARY.md#residual) $r_i = y_i - (m x_i + c)$ at each point. Least squares chooses $m$ and $c$ to minimize the **sum of squared residuals** $S = \sum_i r_i^2$. Setting the two partial derivatives to zero, $\partial S / \partial m = 0$ and $\partial S / \partial c = 0$, gives the **normal equations**, which are exactly a 2x2 system:

$$\begin{bmatrix} \sum x_i^2 & \sum x_i \\ \sum x_i & N \end{bmatrix} \begin{bmatrix} m \\ c \end{bmatrix} = \begin{bmatrix} \sum x_i y_i \\ \sum y_i \end{bmatrix}.$$

In matrix form, stack the data into a **design matrix** $A$ (one row $[x_i, 1]$ per point) and a vector $\mathbf{d}$ of the $y_i$. The normal equations are then

$$(A^{\mathsf{T}} A)\,\mathbf{m} = A^{\mathsf{T}} \mathbf{d},$$

which is the same `np.linalg.solve` you ran in Part 1, now built from 25 data points instead of two hand-written equations.

> **This is the 2x2 system from Part 1, now built from 25 data points instead of two equations.**

In [ ]:
# Build the design matrix A (column of L, column of ones) and data vector d = T^2.
L = pendulum['length_m'].values
d = pendulum['T2'].values
A = np.column_stack([L, np.ones_like(L)])

# Solve the normal equations (A^T A) m = A^T d.
ATA = A.T @ A
ATd = A.T @ d
m_solve = np.linalg.solve(ATA, ATd)
slope_ne, intercept_ne = m_solve

print(f'normal-equations slope:     {slope_ne:.4f} s^2/m')
print(f'normal-equations intercept: {intercept_ne:.4f} s^2')

In [ ]:
# Compare against np.polyfit, then recover g.
slope_pf, intercept_pf = np.polyfit(L, d, 1)
g_matrix = 4 * np.pi**2 / slope_ne

print(f'polyfit slope:     {slope_pf:.4f} s^2/m')
print(f'polyfit intercept: {intercept_pf:.4f} s^2')
print(f'agree to 4 dp:     {np.allclose(m_solve, [slope_pf, intercept_pf], atol=1e-4)}')
print(f'g = 4 pi^2 / slope = {g_matrix:.2f} m/s^2')

The intercept is small but **nonzero** (about -0.033 s²). The ideal pendulum law $T^2 = (4\pi^2/g)\,l$ predicts a zero intercept (zero length, zero period). The small offset is real-apparatus effects (the pivot and the bob are not idealized points) plus fit noise; the slope, which carries $g$, is what we trust.

The best-fit line you did by eye in Week 1 is a 2x2 solve, and it reproduces the same g = 9.66 m/s².

**Optional swap (clearly marked).** To re-fit the live Wk 2 class CSV instead of the sample data, re-point `PENDULUM_URL` in the cell below and re-run Parts 2–5. The in-class file has five columns (`station,trial,length_m,t10_s,period_s`) and 125 rows; every cell here keys on `length_m` and `period_s`, so the code runs unchanged. The self-check numbers above are pinned to the sample CSV, so they will shift; that is expected.

In [ ]:
# OPTIONAL: re-point to the live Wk 2 class CSV (self-checks above will change).
# PENDULUM_URL = 'https://raw.githubusercontent.com/soule-geophysics/geol-333-714/main/data/pendulum_inclass.csv'
# pendulum = pd.read_csv(PENDULUM_URL)
# pendulum['T2'] = pendulum['period_s'] ** 2
# Then re-run the Part 5 fit cells above.
print('Swap cell: edit and uncomment to use the live class CSV.')

**Self-check.** Normal-equations solution: slope **4.0862 s²/m**, intercept **-0.0332 s²**; it must agree with `np.polyfit` to at least 4 decimal places, and g = 4π²/slope = **9.66 m/s²**. If the two methods disagree, check the column order of your design matrix against how you unpack the solution (slope first, then intercept).

## Part 6: Drift preview on the stairwell base ties

Now switch datasets. The stairwell survey reads the gravimeter at ground level, climbs the stairs reading once per landing, then re-reads ground level at the end. The two ground reads are at the *same physical place*, so any difference between them is pure **instrument drift**: spring creep, temperature, and tides nudge the reading over the hour even though Earth's gravity did not change. The cell below isolates the two ground ties, computes the elapsed time, the closure (end minus start), and the drift rate, then corrects all 11 reads.

In [ ]:
# Isolate the two ground-level ties (note the deliberate typo 'gound level').
ground = stairwell[stairwell['station'].isin(['Ground level', 'gound level'])]

start = ground[ground['station'] == 'Ground level'].iloc[0]
end = ground[ground['station'] == 'gound level'].iloc[0]

elapsed = end['minutes_since_start'] - start['minutes_since_start']
closure = end['reading_mgal'] - start['reading_mgal']
drift_rate = closure / elapsed

print(f'elapsed time between ground reads: {elapsed:.0f} minutes')
print(f'closure (end - start):             {closure:+.3f} mGal')
print(f'drift rate:                        {drift_rate:+.4f} mGal/min')

In [ ]:
# Apply the two-point drift correction to ALL 11 reads.
stairwell['drift_corrected'] = (
    stairwell['reading_mgal'] - drift_rate * stairwell['minutes_since_start']
)

# After correction the two ground reads should collapse onto the same value.
print(stairwell[stairwell['station'].isin(['Ground level', 'gound level'])]
      [['station', 'reading_mgal', 'drift_corrected']].to_string(index=False))

Two points determine a line *exactly*, so this drift correction is interpolation: there is no scatter to fit and no residual to minimize. The genuine statistical drift fit comes in **HW2**, whose profile dataset re-occupies the base 23 times; with 23 base reads, drift becomes a real least-squares problem with residuals and an uncertainty.

**Self-check.** Elapsed time between ground reads: **68 minutes**; closure: **+0.077 mGal**; rate: **+0.0011 mGal/min**. If your closure is -0.077 you computed start minus end; if your rate is off by a factor of 60, check that you used minutes, not hours.

## Part 7: Free-air gradient preview

With drift removed, the slope of the drift-corrected reading against **elevation** is the [**free-air gradient**](https://github.com/soule-geophysics/geol-333-714/blob/main/GLOSSARY.md#free-air-gradient): how fast gravity decreases with height above radius $R$ in $g = GM/R^2$. We fit it through the climbing reads (the two ground reads are both at elevation 0 and only anchored the drift), and we use `np.polyfit(..., cov=True)` so we get a 1-sigma uncertainty, not just a single number.

In [ ]:
# Fit the drift-corrected reading vs elevation through the climbing reads.
climb = stairwell[~stairwell['station'].isin(['Ground level', 'gound level'])]

coeffs, cov = np.polyfit(climb['elevation_m'], climb['drift_corrected'], 1, cov=True)
gradient, grad_intercept = coeffs
gradient_sigma = np.sqrt(cov[0, 0])

print(f'free-air gradient: {gradient:.3f} +/- {gradient_sigma:.3f} mGal/m (1 sigma)')
print(f'canonical value:   {-0.3086:.4f} mGal/m')

In [ ]:
# Plot the climbing reads with the fitted line.
# Color AND line dash both distinguish the fit from the data (readable in grayscale).
import plotly.graph_objects as go

elev = climb['elevation_m']
fit_line = gradient * elev + grad_intercept

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=elev, y=climb['drift_corrected'], mode='markers', name='drift-corrected reads',
    marker=dict(size=11, symbol='circle', color=px.colors.qualitative.Safe[0]),
))
fig.add_trace(go.Scatter(
    x=elev, y=fit_line, mode='lines', name='least-squares fit',
    line=dict(dash='dash', color=px.colors.qualitative.Safe[1]),
))
fig.update_layout(
    title='Drift-corrected gravity vs elevation (free-air gradient)',
    xaxis_title='Elevation above ground (m)',
    yaxis_title='Drift-corrected reading (mGal)',
)
fig.show()

**Figure description:** A scatter plot of the drift-corrected gravimeter reading (mGal) against elevation above ground (meters, 0 to ~16), with the least-squares fit overlaid as a dashed line. The reading falls roughly linearly as elevation rises; the fitted slope is the free-air gradient, -0.287 ± 0.008 mGal/m, against the canonical -0.3086 mGal/m. Why the two differ is HW1's graded question, so it is not stated here. The fitted gradient is printed by the cell above, so the comparison does not require reading the plot.

The fitted gradient and the canonical -0.3086 mGal/m do not match. **Why** is HW1's graded question, so do not answer it here; note the gap and that it is larger than the 1-sigma uncertainty.

**Self-check.** Your fitted gradient should land at **-0.287 ± 0.008 mGal/m**, i.e., between about -0.295 and -0.279, against the canonical **-0.3086 mGal/m**. The sign must be negative; if it is positive, you fitted elevation vs reading instead of reading vs elevation. If you got exactly -0.3086, you are looking at the canonical constant, not your fit.

## Part 8: Stairwell procedure recap and HW1 launch

The stairwell survey procedure, in three moves: (1) read the gravimeter at the base (ground level) to start the clock; (2) climb the stairwell, taking one read per landing as you go up; (3) re-read the base at the end. The repeated base ties exist so you can *measure and remove* the drift, the instrument row of the assumptions table. The same dataset supports two framings: reading vs time (drift) and reading vs elevation (free-air). The cell below checks the data against this procedure.

In [ ]:
# Quick sanity walk over the stairwell dataset, as loaded from the source CSV.
# (Re-read fresh so the shape is the as-delivered file, before Part 6 added the
# drift_corrected column to the working copy.)
stairwell_raw = pd.read_csv(STAIRWELL_URL)
print('shape:', stairwell_raw.shape)
stairwell_raw.head()

### HW1: Read the Stairwell

HW1 launches this week and is **due Wed Sep 23, 11:59 PM**. Submit `HW1_LASTNAME.ipynb` to the HW1 dropbox on Brightspace. The rubric is linked in the HW1 assignment instructions.

HW1 asks you to run the full drift-then-free-air procedure you previewed in Parts 6–7 on the stairwell data, report the free-air gradient with its uncertainty, derive the canonical gradient from `g = GM/R²` using the binomial expansion, and explain why the measured gradient differs from the canonical value. Parts 6–8 here rehearse every move HW1 grades.

**GEOL 714 only:** Critical Literature Review Checkpoint 1 is due **next Wed, Sep 23**: post your topic and 3 candidate papers (one sentence each on why each qualifies) to the CLR checkpoint thread on the discussion board.

**Self-check.** `stairwell.csv` loads with shape **(11, 7)**: 9 stairwell stations plus 2 ground-level ties, with 7 columns (station, time, minutes_since_start, elevation_m, reading_mgal, relative_mgal, uncertainty_mgal). If you count only one ground read, note the second one's station label is misspelled `gound level` in the source data; it is preserved from the source data.

## Part 9 (after class): Code critique, three buggy drift snippets

These three snippets were **supplied by the instructor**, and each contains exactly **one planted bug**. Each is self-contained: the data lives in inline lists, so nothing depends on uploads or earlier cells. Reviewing code you did not write, against data you understand, is the skill being practiced here. **You do not operate any AI tool in this exercise.** For each snippet: run it, read the printed output, and in the free-response cell below it name the bug, the symptom you saw, and the one-line fix.

In [ ]:
# SNIPPET 1 (instructor-supplied). Uses the real stairwell ground ties.
# Ground reads: 3423.430 at t = 0 min, 3423.507 at t = 68 min.
t = [0.0, 68.0]
reading = [3423.430, 3423.507]
drift_rate = (reading[1] - reading[0]) / (t[1] - t[0])   # +0.077 / 68 mGal/min

corrected = [r + drift_rate * ti for r, ti in zip(reading, t)]

print(f'drift rate:               {drift_rate:+.4f} mGal/min')
print(f'corrected ground reads:   {corrected[0]:.3f}, {corrected[1]:.3f} mGal')
print(f'they differ by:           {corrected[1] - corrected[0]:.3f} mGal')

**Snippet 1 critique.** Name the bug, the symptom in the printed output, and the one-line fix.

*(Your answer):*

In [ ]:
# SNIPPET 2 (instructor-supplied). Inline base reads spanning two field days
# with an overnight instrument tare between them.
import numpy as np

t = [0, 40, 80, 1440, 1480, 1520]                       # minutes; day 2 starts at 1440
reading = [3423.400, 3423.448, 3423.496,                # day 1
           3423.148, 3423.196, 3423.244]                # day 2 (after overnight tare)

slope, intercept = np.polyfit(t, reading, 1)
print(f'fitted drift rate across all six reads: {slope:+.6f} mGal/min')

**Snippet 2 critique.** Name the bug, the symptom in the printed output, and the one-line fix.

*(Your answer):*

In [ ]:
# SNIPPET 3 (instructor-supplied). Same ground ties as snippet 1.
t = [0.0, 68.0]
reading = [3423.430, 3423.507]
drift_rate = (reading[1] - reading[0]) / (t[1] - t[0])
closure = reading[1] - reading[0]                        # end minus start

# Correct the drift (subtract the drift term)...
corrected = [r - drift_rate * ti for r, ti in zip(reading, t)]
# ...then ALSO subtract the raw closure from the end read 'to tie it back to base'.
corrected[1] = corrected[1] - closure

print(f'corrected start ground read: {corrected[0]:.3f} mGal')
print(f'corrected end ground read:   {corrected[1]:.3f} mGal')
print(f'end minus start:             {corrected[1] - corrected[0]:+.3f} mGal')

**Snippet 3 critique.** Name the bug, the symptom in the printed output, and the one-line fix.

*(Your answer):*

**Self-check.** Each snippet hides exactly one bug. Symptoms: snippet 1's "corrected" ground reads disagree by **0.154 mGal** (exactly twice the drift closure; after your fix they agree exactly); snippet 2's single cross-gap fit reports about **-0.0002 mGal/min** while both per-day fits give **+0.0012 mGal/min** (the overnight tare, not the drift, is driving the fit); snippet 3's corrected end ground read lands **0.077 mGal below** the start read (corrected twice). If you found two bugs in one snippet, one of them is a style choice, not an error.